In [1]:
from __future__ import annotations

import os
from getpass import getpass
from typing import Any

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "SimilarityArticle"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

articles = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="external_id",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="topic",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="difficulty",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content_type",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: SimilarityArticle


In [5]:
article_data = [
    {
        "external_id": "art-001",
        "title": "Weaviate Cloud Authentication",
        "content": (
            "Weaviate Cloud requires a cluster URL, a Weaviate API key and provider headers "
            "such as the OpenAI API key for text vectorization and generative search."
        ),
        "topic": "cloud",
        "difficulty": "beginner",
        "content_type": "guide",
    },
    {
        "external_id": "art-002",
        "title": "Connecting to Weaviate Cloud",
        "content": (
            "To connect to Weaviate Cloud, use the cluster URL, authenticate with the Weaviate API key "
            "and pass the OpenAI API key in provider headers."
        ),
        "topic": "cloud",
        "difficulty": "beginner",
        "content_type": "guide",
    },
    {
        "external_id": "art-003",
        "title": "Vector Search Basics",
        "content": (
            "Vector search uses embeddings to find semantically similar objects. "
            "It is useful when users describe meaning rather than exact keywords."
        ),
        "topic": "vector_search",
        "difficulty": "beginner",
        "content_type": "concept",
    },
    {
        "external_id": "art-004",
        "title": "Semantic Search with Embeddings",
        "content": (
            "Semantic search compares embedding vectors and returns objects that are close in vector space. "
            "This helps match meaning even when words are different."
        ),
        "topic": "vector_search",
        "difficulty": "beginner",
        "content_type": "concept",
    },
    {
        "external_id": "art-005",
        "title": "BM25 Keyword Search",
        "content": (
            "BM25 is a keyword-based search method. It works well for exact terms, product names, "
            "technical identifiers, categories and acronyms."
        ),
        "topic": "bm25",
        "difficulty": "intermediate",
        "content_type": "concept",
    },
    {
        "external_id": "art-006",
        "title": "Hybrid Search with Alpha",
        "content": (
            "Hybrid search combines BM25 keyword search with vector search. "
            "The alpha parameter controls the balance between lexical and semantic ranking."
        ),
        "topic": "hybrid_search",
        "difficulty": "intermediate",
        "content_type": "guide",
    },
    {
        "external_id": "art-007",
        "title": "RAG with Weaviate",
        "content": (
            "Retrieval-Augmented Generation retrieves relevant chunks from Weaviate and passes them "
            "to a generative model to produce grounded answers."
        ),
        "topic": "rag",
        "difficulty": "advanced",
        "content_type": "architecture",
    },
    {
        "external_id": "art-008",
        "title": "Focused RAG over Notebooks",
        "content": (
            "Focused RAG works best when the collection contains clean chunks from relevant notebooks "
            "and Markdown concept notes instead of unrelated project files."
        ),
        "topic": "rag",
        "difficulty": "advanced",
        "content_type": "architecture",
    },
    {
        "external_id": "art-009",
        "title": "Named Vectors",
        "content": (
            "Named vectors allow one object to have multiple vector spaces, for example one vector "
            "for title and another vector for description."
        ),
        "topic": "named_vectors",
        "difficulty": "advanced",
        "content_type": "concept",
    },
    {
        "external_id": "art-010",
        "title": "Multi-Tenancy",
        "content": (
            "Multi-tenancy isolates data for different customers, companies or users inside one shared "
            "collection schema."
        ),
        "topic": "multi_tenancy",
        "difficulty": "advanced",
        "content_type": "architecture",
    },
    {
        "external_id": "art-011",
        "title": "Tenant Isolation in SaaS",
        "content": (
            "Tenant isolation is useful for SaaS applications where each customer should access only "
            "their own data while sharing one common schema."
        ),
        "topic": "multi_tenancy",
        "difficulty": "advanced",
        "content_type": "architecture",
    },
]

In [6]:
result = articles.data.insert_many(article_data)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [7]:
response = articles.query.fetch_objects(
    limit=20,
    return_properties=[
        "external_id",
        "title",
        "topic",
        "difficulty",
        "content_type",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print(obj.properties)
    print("-" * 80)

UUID: 3285f4a1-2f30-443e-8789-743851a402ac
{'title': 'Weaviate Cloud Authentication', 'difficulty': 'beginner', 'topic': 'cloud', 'external_id': 'art-001', 'content_type': 'guide'}
--------------------------------------------------------------------------------
UUID: 6e9c57b7-746b-4961-b4e6-7c29e1c69c55
{'title': 'BM25 Keyword Search', 'difficulty': 'intermediate', 'topic': 'bm25', 'external_id': 'art-005', 'content_type': 'concept'}
--------------------------------------------------------------------------------
UUID: 756cc503-4cba-4b06-8403-a7fe35fa0db9
{'title': 'Tenant Isolation in SaaS', 'difficulty': 'advanced', 'topic': 'multi_tenancy', 'external_id': 'art-011', 'content_type': 'architecture'}
--------------------------------------------------------------------------------
UUID: 789dfcb6-8bbf-44be-a8c3-30363dbd353e
{'title': 'Vector Search Basics', 'difficulty': 'beginner', 'topic': 'vector_search', 'external_id': 'art-003', 'content_type': 'concept'}
---------------------------

In [8]:
def get_article_by_external_id(external_id: str):
    response = articles.query.fetch_objects(
        filters=Filter.by_property("external_id").equal(external_id),
        limit=1,
        return_properties=[
            "external_id",
            "title",
            "content",
            "topic",
            "difficulty",
            "content_type",
        ],
    )

    if not response.objects:
        raise ValueError(f"Article not found: {external_id}")

    return response.objects[0]

In [9]:
base_article = get_article_by_external_id("art-007")

print("Base UUID:", base_article.uuid)
print("Base title:", base_article.properties["title"])
print("Base topic:", base_article.properties["topic"])
print("Base content:", base_article.properties["content"])

Base UUID: 872f67ee-1110-42bb-bc52-5fa487a3fdc0
Base title: RAG with Weaviate
Base topic: rag
Base content: Retrieval-Augmented Generation retrieves relevant chunks from Weaviate and passes them to a generative model to produce grounded answers.


In [10]:
response = articles.query.near_object(
    near_object=base_article.uuid,
    limit=5,
    return_properties=[
        "external_id",
        "title",
        "content",
        "topic",
        "difficulty",
        "content_type",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("External ID:", obj.properties["external_id"])
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 1.1920928955078125e-06
External ID: art-007
Title: RAG with Weaviate
Topic: rag
Content: Retrieval-Augmented Generation retrieves relevant chunks from Weaviate and passes them to a generative model to produce grounded answers.
--------------------------------------------------------------------------------
Distance: 0.41445082426071167
External ID: art-001
Title: Weaviate Cloud Authentication
Topic: cloud
Content: Weaviate Cloud requires a cluster URL, a Weaviate API key and provider headers such as the OpenAI API key for text vectorization and generative search.
--------------------------------------------------------------------------------
Distance: 0.4187626838684082
External ID: art-008
Title: Focused RAG over Notebooks
Topic: rag
Content: Focused RAG works best when the collection contains clean chunks from relevant notebooks and Markdown concept notes instead of unrelated project files.
--------------------------------------------------------------------------------
Di

In [11]:
similar_articles = [
    obj
    for obj in response.objects
    if obj.uuid != base_article.uuid
]

for obj in similar_articles:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("-" * 80)

Distance: 0.41445082426071167
Title: Weaviate Cloud Authentication
Topic: cloud
--------------------------------------------------------------------------------
Distance: 0.4187626838684082
Title: Focused RAG over Notebooks
Topic: rag
--------------------------------------------------------------------------------
Distance: 0.4786165952682495
Title: Connecting to Weaviate Cloud
Topic: cloud
--------------------------------------------------------------------------------
Distance: 0.526250958442688
Title: Vector Search Basics
Topic: vector_search
--------------------------------------------------------------------------------


In [12]:
def find_related_articles(
    external_id: str,
    *,
    limit: int = 5,
    same_topic_only: bool = False,
) -> None:
    base_obj = get_article_by_external_id(external_id)

    filters = None

    if same_topic_only:
        filters = Filter.by_property("topic").equal(base_obj.properties["topic"])

    response = articles.query.near_object(
        near_object=base_obj.uuid,
        filters=filters,
        limit=limit + 1,
        return_properties=[
            "external_id",
            "title",
            "content",
            "topic",
            "difficulty",
            "content_type",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    related = [
        obj
        for obj in response.objects
        if obj.uuid != base_obj.uuid
    ][:limit]

    print("BASE ARTICLE")
    print("External ID:", base_obj.properties["external_id"])
    print("Title:", base_obj.properties["title"])
    print("Topic:", base_obj.properties["topic"])
    print("-" * 80)

    print("RELATED ARTICLES")
    for obj in related:
        print("Distance:", obj.metadata.distance)
        print("External ID:", obj.properties["external_id"])
        print("Title:", obj.properties["title"])
        print("Topic:", obj.properties["topic"])
        print("Difficulty:", obj.properties["difficulty"])
        print("Content type:", obj.properties["content_type"])
        print("Content:", obj.properties["content"])
        print("-" * 80)

In [13]:
find_related_articles(
    "art-007",
    limit=5,
)

BASE ARTICLE
External ID: art-007
Title: RAG with Weaviate
Topic: rag
--------------------------------------------------------------------------------
RELATED ARTICLES
Distance: 0.41445082426071167
External ID: art-001
Title: Weaviate Cloud Authentication
Topic: cloud
Difficulty: beginner
Content type: guide
Content: Weaviate Cloud requires a cluster URL, a Weaviate API key and provider headers such as the OpenAI API key for text vectorization and generative search.
--------------------------------------------------------------------------------
Distance: 0.4187626838684082
External ID: art-008
Title: Focused RAG over Notebooks
Topic: rag
Difficulty: advanced
Content type: architecture
Content: Focused RAG works best when the collection contains clean chunks from relevant notebooks and Markdown concept notes instead of unrelated project files.
--------------------------------------------------------------------------------
Distance: 0.4786165952682495
External ID: art-002
Title: Connec

In [14]:
find_related_articles(
    "art-007",
    limit=5,
    same_topic_only=True,
)

BASE ARTICLE
External ID: art-007
Title: RAG with Weaviate
Topic: rag
--------------------------------------------------------------------------------
RELATED ARTICLES
Distance: 0.4187626838684082
External ID: art-008
Title: Focused RAG over Notebooks
Topic: rag
Difficulty: advanced
Content type: architecture
Content: Focused RAG works best when the collection contains clean chunks from relevant notebooks and Markdown concept notes instead of unrelated project files.
--------------------------------------------------------------------------------


In [15]:
find_related_articles(
    "art-001",
    limit=5,
)

BASE ARTICLE
External ID: art-001
Title: Weaviate Cloud Authentication
Topic: cloud
--------------------------------------------------------------------------------
RELATED ARTICLES
Distance: 0.08361470699310303
External ID: art-002
Title: Connecting to Weaviate Cloud
Topic: cloud
Difficulty: beginner
Content type: guide
Content: To connect to Weaviate Cloud, use the cluster URL, authenticate with the Weaviate API key and pass the OpenAI API key in provider headers.
--------------------------------------------------------------------------------
Distance: 0.41445082426071167
External ID: art-007
Title: RAG with Weaviate
Topic: rag
Difficulty: advanced
Content type: architecture
Content: Retrieval-Augmented Generation retrieves relevant chunks from Weaviate and passes them to a generative model to produce grounded answers.
--------------------------------------------------------------------------------
Distance: 0.5644369125366211
External ID: art-003
Title: Vector Search Basics
Topic: 

In [16]:
find_related_articles(
    "art-010",
    limit=5,
)

BASE ARTICLE
External ID: art-010
Title: Multi-Tenancy
Topic: multi_tenancy
--------------------------------------------------------------------------------
RELATED ARTICLES
Distance: 0.1615578532218933
External ID: art-011
Title: Tenant Isolation in SaaS
Topic: multi_tenancy
Difficulty: advanced
Content type: architecture
Content: Tenant isolation is useful for SaaS applications where each customer should access only their own data while sharing one common schema.
--------------------------------------------------------------------------------
Distance: 0.5978389978408813
External ID: art-009
Title: Named Vectors
Topic: named_vectors
Difficulty: advanced
Content type: concept
Content: Named vectors allow one object to have multiple vector spaces, for example one vector for title and another vector for description.
--------------------------------------------------------------------------------
Distance: 0.6459642648696899
External ID: art-008
Title: Focused RAG over Notebooks
Topic: r

In [17]:
def find_possible_duplicates(
    external_id: str,
    *,
    distance_threshold: float = 0.12,
    limit: int = 5,
) -> None:
    base_obj = get_article_by_external_id(external_id)

    response = articles.query.near_object(
        near_object=base_obj.uuid,
        limit=limit + 1,
        return_properties=[
            "external_id",
            "title",
            "content",
            "topic",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    candidates = [
        obj
        for obj in response.objects
        if obj.uuid != base_obj.uuid and obj.metadata.distance <= distance_threshold
    ]

    print("BASE:")
    print(base_obj.properties["external_id"], "-", base_obj.properties["title"])
    print("-" * 80)

    if not candidates:
        print("No possible duplicates found.")
        return

    print("POSSIBLE DUPLICATES:")
    for obj in candidates:
        print("Distance:", obj.metadata.distance)
        print("External ID:", obj.properties["external_id"])
        print("Title:", obj.properties["title"])
        print("Topic:", obj.properties["topic"])
        print("Content:", obj.properties["content"])
        print("-" * 80)

In [18]:
find_possible_duplicates(
    "art-001",
    distance_threshold=0.18,
)

BASE:
art-001 - Weaviate Cloud Authentication
--------------------------------------------------------------------------------
POSSIBLE DUPLICATES:
Distance: 0.08361470699310303
External ID: art-002
Title: Connecting to Weaviate Cloud
Topic: cloud
Content: To connect to Weaviate Cloud, use the cluster URL, authenticate with the Weaviate API key and pass the OpenAI API key in provider headers.
--------------------------------------------------------------------------------


In [19]:
find_possible_duplicates(
    "art-003",
    distance_threshold=0.18,
)

BASE:
art-003 - Vector Search Basics
--------------------------------------------------------------------------------
POSSIBLE DUPLICATES:
Distance: 0.1107410192489624
External ID: art-004
Title: Semantic Search with Embeddings
Topic: vector_search
Content: Semantic search compares embedding vectors and returns objects that are close in vector space. This helps match meaning even when words are different.
--------------------------------------------------------------------------------


In [20]:
find_possible_duplicates(
    "art-010",
    distance_threshold=0.18,
)

BASE:
art-010 - Multi-Tenancy
--------------------------------------------------------------------------------
POSSIBLE DUPLICATES:
Distance: 0.1615578532218933
External ID: art-011
Title: Tenant Isolation in SaaS
Topic: multi_tenancy
Content: Tenant isolation is useful for SaaS applications where each customer should access only their own data while sharing one common schema.
--------------------------------------------------------------------------------


In [21]:
all_articles_response = articles.query.fetch_objects(
    limit=100,
    return_properties=[
        "external_id",
        "title",
        "topic",
    ],
)

all_articles = all_articles_response.objects

print("Total articles:", len(all_articles))

Total articles: 11


In [22]:
def build_related_map(
    objects,
    *,
    related_limit: int = 3,
) -> dict[str, list[dict[str, Any]]]:
    related_map = {}

    for base_obj in objects:
        response = articles.query.near_object(
            near_object=base_obj.uuid,
            limit=related_limit + 1,
            return_properties=[
                "external_id",
                "title",
                "topic",
            ],
            return_metadata=MetadataQuery(distance=True),
        )

        related = []

        for obj in response.objects:
            if obj.uuid == base_obj.uuid:
                continue

            related.append(
                {
                    "external_id": obj.properties["external_id"],
                    "title": obj.properties["title"],
                    "topic": obj.properties["topic"],
                    "distance": obj.metadata.distance,
                }
            )

        related_map[base_obj.properties["external_id"]] = related[:related_limit]

    return related_map

In [23]:
related_map = build_related_map(
    all_articles,
    related_limit=3,
)

for external_id, related in related_map.items():
    base_title = next(
        obj.properties["title"]
        for obj in all_articles
        if obj.properties["external_id"] == external_id
    )

    print("BASE:", external_id, "-", base_title)

    for item in related:
        print(
            "  ->",
            item["external_id"],
            "|",
            item["title"],
            "| topic:",
            item["topic"],
            "| distance:",
            item["distance"],
        )

    print("-" * 80)

BASE: art-001 - Weaviate Cloud Authentication
  -> art-002 | Connecting to Weaviate Cloud | topic: cloud | distance: 0.08361470699310303
  -> art-007 | RAG with Weaviate | topic: rag | distance: 0.41445082426071167
  -> art-003 | Vector Search Basics | topic: vector_search | distance: 0.5644369125366211
--------------------------------------------------------------------------------
BASE: art-005 - BM25 Keyword Search
  -> art-006 | Hybrid Search with Alpha | topic: hybrid_search | distance: 0.2910425662994385
  -> art-003 | Vector Search Basics | topic: vector_search | distance: 0.4464256763458252
  -> art-004 | Semantic Search with Embeddings | topic: vector_search | distance: 0.4660254716873169
--------------------------------------------------------------------------------
BASE: art-011 - Tenant Isolation in SaaS
  -> art-010 | Multi-Tenancy | topic: multi_tenancy | distance: 0.1615578532218933
  -> art-001 | Weaviate Cloud Authentication | topic: cloud | distance: 0.60386580228805

In [24]:
base_article = get_article_by_external_id("art-007")

response = articles.generate.near_object(
    near_object=base_article.uuid,
    grouped_task=(
        "Explain how the retrieved articles are related to the base article about RAG. "
        "Use only the retrieved articles. "
        "Mention the main shared concepts."
    ),
    limit=5,
    return_properties=[
        "external_id",
        "title",
        "content",
        "topic",
        "difficulty",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print("-", obj.properties["external_id"], "|", obj.properties["title"], "|", obj.properties["topic"])

The retrieved articles are related to the base article about Retrieval-Augmented Generation (RAG) by discussing the architecture, implementation, and foundational concepts that support RAG systems, particularly those that utilize Weaviate for retrieval and generation processes. 

The main shared concepts include:

1. **Connection to Weaviate**: Articles such as "RAG with Weaviate" and "Connecting to Weaviate Cloud" emphasize the importance of authenticating and connecting to Weaviate, which is central to the retrieval component of RAG. Proper authentication, including the use of API keys, is crucial for leveraging Weaviate's capabilities in a RAG framework.

2. **Retrieval Mechanism**: The article "RAG with Weaviate" directly addresses how RAG systems leverage retrieved chunks of information to provide grounded answers, highlighting the contextual relevance brought about by the retrieval process. This mechanism is further elaborated in "Focused RAG over Notebooks," which suggests that 

In [25]:
def explain_related_content(
    external_id: str,
    *,
    limit: int = 5,
) -> None:
    base_obj = get_article_by_external_id(external_id)

    response = articles.generate.near_object(
        near_object=base_obj.uuid,
        grouped_task=(
            "Explain how the retrieved articles are related to the base article. "
            "Use only the retrieved articles. "
            "Keep the explanation concise and practical."
        ),
        limit=limit + 1,
        return_properties=[
            "external_id",
            "title",
            "content",
            "topic",
            "difficulty",
        ],
    )

    print("BASE ARTICLE:")
    print(base_obj.properties["external_id"], "-", base_obj.properties["title"])
    print("-" * 80)

    print("GENERATED EXPLANATION:")
    print(response.generated)

    print("\nSOURCES:")
    for obj in response.objects:
        if obj.uuid == base_obj.uuid:
            continue

        print(
            "-",
            obj.properties["external_id"],
            "|",
            obj.properties["title"],
            "|",
            obj.properties["topic"],
        )

In [26]:
explain_related_content(
    "art-010",
    limit=5,
)

BASE ARTICLE:
art-010 - Multi-Tenancy
--------------------------------------------------------------------------------
GENERATED EXPLANATION:
The retrieved articles are related to the base article primarily through the themes of data management and architecture in multi-tenant systems:

1. **Multi-Tenancy** (art-010) and **Tenant Isolation in SaaS** (art-011) both focus on the concept of separating user data within a shared environment, which is crucial for Software as a Service (SaaS) applications. They explore how to effectively isolate tenants' data while still using a common schema, enhancing security and privacy.

2. While not directly linked to multi-tenancy, **Named Vectors** (art-009) introduces the concept of different vector spaces for an object, which could complement multi-tenancy architectures by allowing tailored data representations for different tenants.

3. **Focused RAG over Notebooks** (art-008) discusses optimizing data retrieval strategies, which can be relevant fo

In [27]:
client.close()